# 15.2 몬테카를로 트리 탐색: 이론과 실습

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SuminHan/book-ml/blob/main/notebooks/ml2/chapter15_2_mcts_nim.ipynb)

책 본문: [Chapter 15.2](https://smhanlab.com/book-ml/kor/ml2/chapter15/2.html)

선택-확장-시뮬레이션-역전파 4단계를 그대로 구현해, 승패를 수식으로
알 수 있는 님(Nim) 게임에서 MCTS가 필승 전략("상대에게 4의 배수를
남긴다")을 스스로 찾아내는지 확인합니다. 마지막 섹션에서는 본문
"버그 투어"의 네 가지 실수를 실제로 만들어 그 증상을 확인합니다.

In [1]:
import matplotlib
matplotlib.use("Agg")
from matplotlib import font_manager
import matplotlib.pyplot as plt
kr = [f.name for f in font_manager.fontManager.ttflist if "Noto Sans CJK KR" in f.name]
if kr: plt.rcParams["font.sans-serif"] = [kr[0]]
plt.rcParams["axes.unicode_minus"] = False
import math
import random
import subprocess
from pathlib import Path

IMG = "/home/smhan/book-ml/kor/src/images"

def svg_name(fname):
    return IMG + "/" + fname

## 1. 님 게임 규칙

돌을 1~3개씩 번갈아 가져가고, **마지막 돌을 가져가는 쪽이 이긴다**.
돌이 5개 남았을 때, "상대에게 4개를 남기는" 1개 가져가기가 필승 수다.
(상대가 \(m\)개를 가져가면 내가 \(4-m\)개를 가져가면 된다 — 2수짜리 강제 구조.)

In [2]:
def nim_moves(state):
    """이 상태에서 가능한 수: 남은 돌 수만큼, 1개/2개/3개 가져가기."""
    return list(range(1, min(3, state) + 1))

def nim_step(state, move, player):
    """move개를 가져간다: 남은 돌이 줄고, 상대 차례가 된다."""
    return max(0, state - move), -player

def nim_terminal(state):
    return state == 0

def nim_rollout(state, player, rng):
    """시뮬레이션(롤아웃): 완전히 무작위 정책으로 끝까지. 승자(마지막
    돌을 가져간 플레이어) 반환. 이미 종료된 상태면 막 이동한 쪽이 승자."""
    if nim_terminal(state):
        return -player
    s, p = state, player
    while not nim_terminal(s):
        s, p = nim_step(s, rng.choice(nim_moves(s)), p)
    return -p

## 2. MCTS 구현 (본문 코드 그대로)

노드는 **그 노드에서 "이동한 플레이어"(mover)의 관점**의 통계를
담는다 — 역전파가 무엇을 어떻게 더하는지가 여기서 결정된다.

In [3]:
class MCTSNode:
    def __init__(self, state, player, parent=None):
        self.state = state        # 상태
        self.player = player      # 이 상태의 차례인 플레이어
        self.parent = parent      # 부모 노드 (역전파를 타고 올라갈 용도)
        self.children = {}        # {move: 자식 노드}
        self.visits = 0           # 방문 횟수
        self.wins = 0.0           # "이 노드로 두어 도달한 플레이어"의 승 수

def mcts_nim(stones, iters, c=1.4, seed=0):
    """돌이 stones개인 님 게임에서 루트 위치의 MCTS. 루트 노드 반환."""
    rng = random.Random(seed)
    root = MCTSNode(stones, player=1, parent=None)
    for _ in range(iters):
        # 1) 선택: "아직 다 확장되지 않은 리프"까지 UCT로 내려간다.
        node = root
        while not nim_terminal(node.state) and len(node.children) == len(nim_moves(node.state)):
            node = max(node.children.values(), key=lambda ch:
                       ch.wins / ch.visits + c * math.sqrt(math.log(node.visits) / ch.visits)
                       if ch.visits else float('inf'))
        # 2) 확장: 안 가본 수 하나를 자식으로 추가. (이미 끝난 상태면 확장 생략)
        if not nim_terminal(node.state):
            unexpanded = [m for m in nim_moves(node.state) if m not in node.children]
            m = rng.choice(unexpanded)
            next_state, next_player = nim_step(node.state, m, node.player)
            node.children[m] = MCTSNode(next_state, next_player, parent=node)
            leaf = node.children[m]
        else:
            leaf = node
        # 3) 시뮬레이션: 리프에서 무작위 정책으로 끝까지, 승자 판정
        winner = nim_rollout(leaf.state, leaf.player, rng)
        # 4) 역전파: 지나온 모든 노드 방문 +1; 승자가 mover면 승 +1
        cur = leaf
        while cur is not None:
            cur.visits += 1
            mover = cur.parent.player if cur.parent is not None else cur.player
            cur.wins += 1.0 if winner == mover else 0.0
            cur = cur.parent
    return root

### UCT 선택 규칙 — 숫자로

선택 단계의 점수는 UCB1과 정확히 같은 형태: \(\text{UCT}(s,a) = Q/N + c\sqrt{\ln N(s)/N}\).
본문의 예: 부모 16회 방문, \(A:(7,10),\ B:(3,5),\ C:(0,1)\), \(c=1.4\).

In [4]:
def uct_select(node_children, parent_visits, c=1.4):
    # node_children: {move: (wins, visits)}
    # UCB1과 정확히 같은 형태 -- Chapter 2.2를 그대로 게임 트리에 적용한 것
    return max(node_children, key=lambda m: node_children[m][0] / node_children[m][1] +
               c * math.sqrt(math.log(parent_visits) / node_children[m][1]))

children = {'A': (7, 10), 'B': (3, 5), 'C': (0, 1)}
for m, (q, n) in sorted(children.items()):
    uct = q / n + 1.4 * math.sqrt(math.log(16) / n)
    print(f"{m}: {q}/{n} -> UCT = {uct:.3f}")
print("선택:", uct_select(children, 16, c=1.4), " (승률 0인 C가 선택된다 — 아직 1번밖에 안 시도했으므로)")

A: 7/10 -> UCT = 1.437
B: 3/5 -> UCT = 1.643
C: 0/1 -> UCT = 2.331
선택: C  (승률 0인 C가 선택된다 — 아직 1번밖에 안 시도했으므로)


## 3. 실습 1: 님(5)에서 MCTS로 필승 전략 찾아내기

돌 5개, 3000회 시뮬레이션(seed 0). 예상: "1개 가져가기"가 압도적으로
많이 방문되고, 승률도 가장 높아야 한다.

In [5]:
def report(root, iters, title):
    print(title)
    best = None
    for move, child in sorted(root.children.items()):
        w = child.wins / child.visits if child.visits else 0.0
        share = 100.0 * child.visits / iters
        print(f"  {move}개 가져가기: 승률={w:.3f}, 방문={child.visits}/{iters} ({share:.1f}%)")
        if best is None or child.visits > best[1].visits:
            best = (move, child)
    return best

root5 = mcts_nim(stones=5, iters=3000, seed=0)
best = report(root5, 3000, "돌 5개, 3000회, seed 0")
assert best[0] == 1, f"필승 수 1이 아니라 {best[0]}가 1위"
print(f"\n=> 1위가 '{best[0]}개 가져가기' — 수학적 필승 전략(상대에게 4의 배수)과 일치")

돌 5개, 3000회, seed 0
  1개 가져가기: 승률=0.987, 방문=2954/3000 (98.5%)
  2개 가져가기: 승률=0.269, 방문=26/3000 (0.9%)
  3개 가져가기: 승률=0.150, 방문=20/3000 (0.7%)

=> 1위가 '1개 가져가기' — 수학적 필승 전략(상대에게 4의 배수)과 일치


### 시드 안정성

시드를 0~4로 바꿔가며 5회 반복해도 같은 수가 1위인지 확인합니다.

In [6]:
for seed in range(5):
    r = mcts_nim(5, 3000, seed=seed)
    top = max(r.children.values(), key=lambda c: c.visits)
    print(f"seed {seed}: 1위 = {5 - top.state}개 가져가기 (방문 {top.visits}/3000)")

seed 0: 1위 = 1개 가져가기 (방문 2954/3000)
seed 1: 1위 = 1개 가져가기 (방문 2953/3000)
seed 2: 1위 = 1개 가져가기 (방문 2956/3000)
seed 3: 1위 = 1개 가져가기 (방문 2956/3000)
seed 4: 1위 = 1개 가져가기 (방문 2953/3000)


## 4. 왜 67%를 뛰어넘는가: 무작위 기준선과의 비교

MCTS 없이는 "1개 가져가기"의 값이 **무작위 상대 승률**
\(1-P(4)=2/3\approx67\%\)로밖에 구분되지 않는다(\(P(n)\): 돌 n개에서
차례인 쪽의 무작위 승률). MCTS는 98%대로 이를 뛰어넘는다 —
"상대 대응을 최적 응답으로 읽는" 정보가 트리에 들어갔기 때문이다.

In [7]:
# 무작위 상대 기준선 (모의측정): 한 수를 둔 뒤 남은 게임을 양쪽 무작위로 끝냄
def random_winrate_after(stones, move, seed=0, n=20000):
    rng = random.Random(seed)
    wins = 0
    for _ in range(n):
        s, p = nim_step(stones, move, 1)   # 내가 move개 가져감 -> 상대 차례
        winner = nim_rollout(s, p, rng)
        wins += (winner == 1)
    return wins / n

print("돌 5개에서 각 수의 무작위 상대 승률 (모의측정):")
for m in nim_moves(5):
    print(f"  {m}개 가져가기: {random_winrate_after(5, m):.3f}")

w1 = root5.children[1].wins / root5.children[1].visits
print(f"\nMCTS 추정(1개 수): {w1:.3f}  vs  무작위 기준선 0.667")
print(f"  -> MCTS가 무작위 기준선을 {w1 - 0.667:+.1%}만큼 뛰어넘는다")

돌 5개에서 각 수의 무작위 상대 승률 (모의측정):
  1개 가져가기: 0.671
  2개 가져가기: 0.499
  3개 가져가기: 0.499

MCTS 추정(1개 수): 0.987  vs  무작위 기준선 0.667
  -> MCTS가 무작위 기준선을 +32.0%만큼 뛰어넘는다


## 5. 수렴 곡선 (시드 0)

이기는 수("1개 가져가기")의 방문 점유율과 추정 승률을 시뮬레이션
횟수(로그축)에 대해 추적합니다. 본문 그래프
`ch15_2_mcts_convergence.svg`도 이 데이터로 만들어집니다.

In [8]:
import numpy as np
import matplotlib.pyplot as plt

def conv_trace(stones, max_iters=10000, checkpoints=(100, 200, 500, 1000, 2000, 5000, 10000), seed=0):
    """시뮬레이션을 한 번만 돌리며 체크포인트마다 루트 통계를 찍는다."""
    rng = random.Random(seed)
    root = MCTSNode(stones, player=1, parent=None)
    c = 1.4
    cps = set(checkpoints)
    for it in range(1, max_iters + 1):
        node = root
        while not nim_terminal(node.state) and len(node.children) == len(nim_moves(node.state)):
            node = max(node.children.values(), key=lambda ch:
                       ch.wins / ch.visits + c * math.sqrt(math.log(node.visits) / ch.visits)
                       if ch.visits else float('inf'))
        if not nim_terminal(node.state):
            unexpanded = [m for m in nim_moves(node.state) if m not in node.children]
            m = rng.choice(unexpanded)
            ns, np_ = nim_step(node.state, m, node.player)
            node.children[m] = MCTSNode(ns, np_, parent=node)
            leaf = node.children[m]
        else:
            leaf = node
        winner = nim_rollout(leaf.state, leaf.player, rng)
        cur = leaf
        while cur is not None:
            cur.visits += 1
            mover = cur.parent.player if cur.parent is not None else cur.player
            cur.wins += 1.0 if winner == mover else 0.0
            cur = cur.parent
        if it in cps:
            out = {}
            for mv, ch in root.children.items():
                out[mv] = (ch.visits, ch.wins / ch.visits if ch.visits else 0.0)
            yield it, out

share1, share3, rate1, rate3 = [], [], [], []
iters_done = []
for it, out in conv_trace(5):
    iters_done.append(it)
    share1.append(100.0 * out[1][0] / it)
    share3.append(100.0 * out[3][0] / it)
    rate1.append(out[1][1])
    rate3.append(out[3][1])

print("  횟수 | 1개 수 점유율 | 1개 수 승률 | 3개 수 점유율 | 3개 수 승률")
for it, s1, s3, r1, r3 in zip(iters_done, share1, share3, rate1, rate3):
    print(f"  {it:5d} |   {s1:5.1f}%     |   {r1:.2f}     |    {s3:4.1f}%    |   {r3:.3f}")

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
axes[0].plot(iters_done, share1, 'o-', color='tab:blue')
axes[0].axhline(2/3, ls='--', color='gray', lw=1)
axes[0].text(110, 67, 'true win rate vs. a random opponent: 2/3', fontsize=9, color='gray')
axes[0].set_xscale('log')
axes[0].set_xlabel('Number of simulations')
axes[0].set_ylabel('Visit share (%)')
axes[0].set_title('(a) Visit share of the winning move ("1")')
axes[0].grid(alpha=0.3)
axes[1].plot(iters_done, rate1, 's-', color='tab:blue', label='Move 1')
axes[1].plot(iters_done, rate3, '^--', color='tab:orange', label='Move 3')
axes[1].axhline(2/3, ls=':', color='gray', lw=1)
axes[1].axhline(1/3, ls=':', color='gray', lw=1)
axes[1].set_xscale('log')
axes[1].set_xlabel('Number of simulations')
axes[1].set_ylabel('Estimated win rate')
axes[1].set_title('(b) Estimated win rate per move (dotted: true rate vs. random)')
axes[1].legend()
axes[1].grid(alpha=0.3)
fig.tight_layout()
fig.savefig(svg_name("ch15_2_mcts_convergence.svg"), bbox_inches="tight")
plt.close(fig)
print("\n저장:", svg_name("ch15_2_mcts_convergence.svg"))

  횟수 | 1개 수 점유율 | 1개 수 승률 | 3개 수 점유율 | 3개 수 승률
    100 |    76.0%     |   0.84     |    10.0%    |   0.200
    200 |    86.0%     |   0.90     |     6.0%    |   0.167
    500 |    93.6%     |   0.95     |     2.8%    |   0.143
   1000 |    96.2%     |   0.97     |     1.5%    |   0.133
   2000 |    97.8%     |   0.98     |     0.9%    |   0.158
   5000 |    99.0%     |   0.99     |     0.4%    |   0.143
  10000 |    99.5%     |   1.00     |     0.2%    |   0.130



저장: /home/smhan/book-ml/kor/src/images/ch15_2_mcts_convergence.svg


## 6. 실습 2: 더 큰 국면(14개)과 탐색 상수 \(c\)

돌 14개면 "상대에게 12(4의 배수)를 남기는" **2개 가져가기**가 필승 수다.
그런데 추정 승률이 5개 국면(98.7%)보다 낮아진다 — 남은 게임이 길어
"트리가 기억할 수 있는" 2수짜리 구조의 비중이 작아져 수렴이 느리다.

In [9]:
root14 = mcts_nim(stones=14, iters=5000, seed=0)
report(root14, 5000, "돌 14개, 5000회, seed 0")
assert max(root14.children.values(), key=lambda c: c.visits).state == 12

# 5만 회까지 늘리면 어디까지 올라가는가
root14b = mcts_nim(stones=14, iters=50000, seed=0)
w14 = root14b.children[2].wins / root14b.children[2].visits
print(f"\n5만 회로 늘리면 2개 수 승률: {w14:.3f}  (천장이 낮은 게 아니라 도달이 느린 것)")

print("\n탐색 상수 c의 효과 (돌 5개, 2000회, seed 0):")
print("   c | 1개 수 승률/방문 | 2개 수 승률/방문 | 3개 수 승률/방문")
for c in (0.8, 1.4, 2.5):
    r = mcts_nim(5, 2000, c=c, seed=0)
    row = []
    for m in (1, 2, 3):
        ch = r.children[m]
        row.append(f"{ch.wins/ch.visits:.3f}/{ch.visits}")
    print(f" {c:3} |   {row[0]:>14} |   {row[1]:>14} |   {row[2]:>14}")
print("  -> c를 키울수록 확실히 나쁜 수에도 방문이 배분된다 (탐험 예산의 다이얼)")

돌 14개, 5000회, seed 0
  1개 가져가기: 승률=0.488, 방문=498/5000 (10.0%)
  2개 가져가기: 승률=0.830, 방문=4198/5000 (84.0%)
  3개 가져가기: 승률=0.441, 방문=304/5000 (6.1%)



5만 회로 늘리면 2개 수 승률: 0.978  (천장이 낮은 게 아니라 도달이 느린 것)

탐색 상수 c의 효과 (돌 5개, 2000회, seed 0):
   c | 1개 수 승률/방문 | 2개 수 승률/방문 | 3개 수 승률/방문
 0.8 |       0.992/1983 |         0.300/10 |          0.143/7
 1.4 |       0.982/1956 |         0.280/25 |         0.158/19
 2.5 |       0.954/1879 |         0.275/69 |         0.154/52
  -> c를 키울수록 확실히 나쁜 수에도 방문이 배분된다 (탐험 예산의 다이얼)


## 7. 실습 3: 틱택토의 첫 수 (15.1절 완전탐색과 연결)

15.1절에서 완전탐색(549,946개 국면)으로 "최선이면 무승부"를 구했던
틱택토를 이번엔 MCTS로: 빈판에서 5000회 시뮬레이션 후 첫 수 9개 비교.
**가운데**가 방문 점유율과 승률 둘 다에서 1위여야 한다.

In [10]:
TIC = [(0, 0), (0, 1), (0, 2), (1, 0), (1, 1), (1, 2), (2, 0), (2, 1), (2, 2)]

def tic_terminal(bd):
    """승자(1 또는 -1), 무승부(0), 진행 중(None)."""
    L = [bd[i*3:(i+1)*3] for i in range(3)]                  # rows
    C = [[bd[r*3+c] for r in range(3)] for c in range(3)]    # cols
    D = [bd[i*3+i] for i in range(3)]
    lines = L + C + [D, [bd[0], bd[4], bd[8]], [bd[2], bd[4], bd[6]]]
    for line in lines:
        if line[0] != 0 and line[0] == line[1] == line[2]:
            return line[0]
    if 0 not in bd:
        return 0
    return None

def tic_rollout(bd, player, rng):
    b = list(bd)
    p = player
    while True:
        t = tic_terminal(b)
        if t is not None:
            return t
        empties = [i for i, x in enumerate(b) if x == 0]
        i = rng.choice(empties)
        b[i] = p
        p = -p

class TicMCTS:
    def __init__(self, bd, player, parent=None, move=None):
        self.bd = tuple(bd)
        self.player = player
        self.parent = parent
        self.move = move
        self.children = {}
        self.visits = 0
        self.wins = 0.0

def tic_mcts(iters, c=1.4, seed=0):
    """빈판 틱택토 루트의 MCTS. 무승부는 승 0.5/패 0으로 환산한다."""
    rng = random.Random(seed)
    root = TicMCTS([0]*9, 1)
    for _ in range(iters):
        node = root
        while tic_terminal(node.bd) is None:
            empties = [i for i, x in enumerate(node.bd) if x == 0]
            if len(node.children) == len(empties):
                node = max(node.children.values(), key=lambda ch:
                           ch.wins / ch.visits + c * math.sqrt(math.log(node.visits) / ch.visits)
                           if ch.visits else float('inf'))
            else:
                break
        t = tic_terminal(node.bd)
        if t is None:
            empties = [i for i, x in enumerate(node.bd) if x == 0]
            i = rng.choice(empties)
            b = list(node.bd); b[i] = node.player
            child = TicMCTS(b, -node.player, parent=node, move=i)
            node.children[i] = child
            leaf = child
        else:
            leaf = node
        winner = t if t is not None else tic_rollout(leaf.bd, leaf.player, rng)
        cur = leaf
        while cur is not None:
            cur.visits += 1
            if cur.parent is not None:
                # mover = 이 노드로 둔 플레이어 = cur.parent.player (님 버전과 같은 관점)
                cur.wins += 1.0 if winner == cur.parent.player else 0.0
            else:
                cur.wins += 1.0 if winner == 1 else (0.5 if winner == 0 else 0.0)  # 루트(미사용): 승 1 / 무 0.5 / 패 0
            cur = cur.parent
    return root

results = {}
for seed in range(3):
    r = tic_mcts(5000, c=1.4, seed=seed)
    rows = []
    for i, ch in r.children.items():
        w = ch.wins / ch.visits if ch.visits else 0.0
        rows.append((ch.visits, i, w))
    rows.sort(reverse=True)
    top_i, top_w = rows[0][1], rows[0][2]
    top_share = rows[0][0] / 5000
    results[seed] = (top_i, top_share, top_w)
    # TIC[i] = (행, 열) 좌표; 인덱스 4 = (1,1) = 가운데
    print(f"시드 {seed} (5000회): 1위 = 좌표 {TIC[top_i]} (가운데) — 승률 {top_w:.3f}, 방문 {rows[0][0]} ({top_share:.0%})")

assert all(res[0] == 4 for res in results.values()), "세 시드 모두 1위가 가운데(4)여야 함"
print("\n=> 세 시드 모두 가운데가 1위 — 15.1절 완전탐색('무승부')과 다른 질문의 답으로 읽는다:")
print("   MCTS의 승률은 '상대가 무작위일 때'의 실전 승률(가운데 > 귀퉁이 > 변)")

시드 0 (5000회): 1위 = 좌표 (1, 1) (가운데) — 승률 0.678, 방문 1876 (38%)


시드 1 (5000회): 1위 = 좌표 (1, 1) (가운데) — 승률 0.682, 방문 1804 (36%)


시드 2 (5000회): 1위 = 좌표 (1, 1) (가운데) — 승률 0.688, 방문 1905 (38%)

=> 세 시드 모두 가운데가 1위 — 15.1절 완전탐색('무승부')과 다른 질문의 답으로 읽는다:
   MCTS의 승률은 '상대가 무작위일 때'의 실전 승률(가운데 > 귀퉁이 > 변)


## 8. 버그 투어: 실제로 만들어 보는 네 가지 실수

본문 "버그 투어"의 네 실수를 코드에서 실제로 만들어 그 증상을
확인합니다 — "에러 없이 돌면서 결과가 이상하다"는 형태가 얼마나
사악한지 체감하는 것이 목적입니다. (참고: 이 4개 변형은 *의도적으로*
나쁘게 쓴 것입니다.)

In [11]:
def mcts_nim_bug1(stones, iters, c=1.4, seed=0):
    """버그 1: 정지 조건이 `while node.children:` — 부분 확장 노드에서
    UCT로 계속 내려가, 첫 번째로 열린 자식에 모든 방문이 집중된다."""
    rng = random.Random(seed)
    root = MCTSNode(stones, player=1, parent=None)
    for _ in range(iters):
        node = root
        while not nim_terminal(node.state) and node.children:
            node = max(node.children.values(), key=lambda ch:
                       ch.wins / ch.visits + c * math.sqrt(math.log(node.visits) / ch.visits)
                       if ch.visits else float('inf'))
        if not nim_terminal(node.state):
            unexpanded = [m for m in nim_moves(node.state) if m not in node.children]
            if unexpanded:
                m = rng.choice(unexpanded)
                ns, np_ = nim_step(node.state, m, node.player)
                node.children[m] = MCTSNode(ns, np_, parent=node)
                leaf = node.children[m]
            else:
                leaf = node
        else:
            leaf = node
        winner = nim_rollout(leaf.state, leaf.player, rng)
        cur = leaf
        while cur is not None:
            cur.visits += 1
            mover = cur.parent.player if cur.parent is not None else cur.player
            cur.wins += 1.0 if winner == mover else 0.0
            cur = cur.parent
    return root

def mcts_nim_bug2(stones, iters, c=1.4, seed=0):
    """버그 2: 역전파에서 승을 '차례인 플레이어'(cur.player)에게 더함 —
    UCT가 상대에게 유리한 수를 좋은 수로 착각한다."""
    rng = random.Random(seed)
    root = MCTSNode(stones, player=1, parent=None)
    for _ in range(iters):
        node = root
        while not nim_terminal(node.state) and len(node.children) == len(nim_moves(node.state)):
            node = max(node.children.values(), key=lambda ch:
                       ch.wins / ch.visits + c * math.sqrt(math.log(node.visits) / ch.visits)
                       if ch.visits else float('inf'))
        if not nim_terminal(node.state):
            unexpanded = [m for m in nim_moves(node.state) if m not in node.children]
            m = rng.choice(unexpanded)
            ns, np_ = nim_step(node.state, m, node.player)
            node.children[m] = MCTSNode(ns, np_, parent=node)
            leaf = node.children[m]
        else:
            leaf = node
        winner = nim_rollout(leaf.state, leaf.player, rng)
        cur = leaf
        while cur is not None:
            cur.visits += 1
            cur.wins += 1.0 if winner == cur.player else 0.0   # <-- 버그: cur.player
            cur = cur.parent
    return root

def mcts_nim_bug3(stones, iters, c=1.4, seed=0):
    """버그 3: 승을 아예 안 더하고 방문만 더함 — Q=0이라 UCT가 탐험 항만으로
    동작하고, 방문이 정확히 균등(각 1/3)해진다."""
    rng = random.Random(seed)
    root = MCTSNode(stones, player=1, parent=None)
    for _ in range(iters):
        node = root
        while not nim_terminal(node.state) and len(node.children) == len(nim_moves(node.state)):
            node = max(node.children.values(), key=lambda ch:
                       ch.wins / ch.visits + c * math.sqrt(math.log(node.visits) / ch.visits)
                       if ch.visits else float('inf'))
        if not nim_terminal(node.state):
            unexpanded = [m for m in nim_moves(node.state) if m not in node.children]
            m = rng.choice(unexpanded)
            ns, np_ = nim_step(node.state, m, node.player)
            node.children[m] = MCTSNode(ns, np_, parent=node)
            leaf = node.children[m]
        else:
            leaf = node
        winner = nim_rollout(leaf.state, leaf.player, rng)
        cur = leaf
        while cur is not None:
            cur.visits += 1          # <-- wins 갱신 누락
            cur = cur.parent
    return root

print("버그 1 (선택 정지 조건 누락, 3000회):")
r = mcts_nim_bug1(5, 3000, seed=0)
print("  방문:", {m: ch.visits for m, ch in sorted(r.children.items())})
print("  -> 첫 번째로 열린 자식 하나에 몰린다 (UCT가 사실상 작동하지 않음)")

print("버그 2 (역전파 관점 반전, 3000회):")
r = mcts_nim_bug2(5, 3000, seed=0)
print("  방문:", {m: ch.visits for m, ch in sorted(r.children.items())})
print("  승률:", {m: round(ch.wins/ch.visits, 3) for m, ch in sorted(r.children.items())})
print("  -> 방문이 거의 균등, 승률이 의미 없는 값: 신호가 완전히 사라짐")

print("버그 3 (승을 안 더하고 방문만, 3000회):")
r = mcts_nim_bug3(5, 3000, seed=0)
print("  방문:", {m: ch.visits for m, ch in sorted(r.children.items())})
print("  -> 정확히 1/3씩: UCB의 균등 분배 성질(Chapter 2.2) 그대로 재현")

print("버그 4 (터미널에서 확장 시도):")
try:
    node = MCTSNode(0, 1)   # 돌 0개 = 터미널
    m = random.Random(0).choice(nim_moves(node.state))
except Exception as e:
    print(f"  {type(e).__name__}: {e}")
    print("  -> 터미널 검사 없이 확장을 시도하면 여기서 크래시")

버그 1 (선택 정지 조건 누락, 3000회):
  방문: {2: 3000}
  -> 첫 번째로 열린 자식 하나에 몰린다 (UCT가 사실상 작동하지 않음)
버그 2 (역전파 관점 반전, 3000회):
  방문: {1: 1072, 2: 1037, 3: 891}
  승률: {1: 0.025, 2: 0.023, 3: 0.013}
  -> 방문이 거의 균등, 승률이 의미 없는 값: 신호가 완전히 사라짐
버그 3 (승을 안 더하고 방문만, 3000회):
  방문: {1: 1000, 2: 1000, 3: 1000}
  -> 정확히 1/3씩: UCB의 균등 분배 성질(Chapter 2.2) 그대로 재현
버그 4 (터미널에서 확장 시도):
  IndexError: Cannot choose from an empty sequence
  -> 터미널 검사 없이 확장을 시도하면 여기서 크래시


## 9. 그림: MCTS의 네 단계 (본문 그림 재현)

왼쪽: 님(5)의 게임 트리 — 루트의 3갈림과 각 자식의 방문/승률(실측 값).
오른쪽: 선택→확장→시뮬레이션→역전파의 순환. Graphviz(`dot`)로
`ch15_2_mcts_loop.svg`를 만듭니다.

In [12]:
# 실측 값으로 왼쪽 트리 채우기 (노드 라벨은 *자식 상태*의 돌 수로)
r5 = mcts_nim(5, 3000, seed=0)
stats = {5 - m: (ch.visits, ch.wins / ch.visits) for m, ch in r5.children.items()}

def node_label(state):
    return f'[label="{state} stones\\n{stats[state][0]} visits / WR {stats[state][1]:.3f}"]'

DOT_FONT = "Noto Sans CJK KR"
HAVE_KR_FONT = any(DOT_FONT in ln for ln in subprocess.run(["fc-list"], capture_output=True, text=True).stdout.splitlines())
F = f' fontname="{DOT_FONT}"' if HAVE_KR_FONT else ""

dot = f"""digraph mcts {{
  rankdir=LR;
  compound=true; label=""; nodesep=0.35; ranksep=0.55;
  graph [{F.strip()}];
  node  [shape=box, style="rounded,filled", fillcolor="#eef2ff"{F}, fontsize=11];
  edge  [{F.strip()}, fontsize=10];
  subgraph cluster_tree {{
    label="Nim (5) game tree (after 3000 simulations)";
    labeljust="l"; fontsize=12;
    root [label="5 stones (my turn)\\n[root]", fillcolor="#dbeafe"];
    n4   {node_label(4)};
    n3   {node_label(3)};
    n2   {node_label(2)};
    root -> n4 [label="take 1"];
    root -> n3 [label="take 2"];
    root -> n2 [label="take 3"];
  }}
  subgraph cluster_loop {{
    label="The four-step cycle";
    labeljust="l"; fontsize=12;
    sel   [label="1. Selection\\nUCT down to a leaf"];
    exp   [label="2. Expansion\\nadd one untried move"];
    sim   [label="3. Simulation\\nrandom playout to the end"];
    back  [label="4. Backpropagation\\nupdate statistics (visits+1, wins update)"];
    sel -> exp -> sim -> back -> sel;
  }}
}}"""

Path("/tmp/ch15_2_mcts_loop.dot").write_text(dot)
subprocess.run(["dot", "-Tsvg", "/tmp/ch15_2_mcts_loop.dot",
                "-o", svg_name("ch15_2_mcts_loop.svg")], check=True)
print("저장:", svg_name("ch15_2_mcts_loop.svg"))

저장: /home/smhan/book-ml/kor/src/images/ch15_2_mcts_loop.svg
